## Exercises on Exploration vs Exploitation (Multi-Armed Bandits)

These paper-and-pencil exercises reinforce Chapter 04: incremental action-value estimation, the sample-average and constant step-size updates, the exponential-recency weighting, the stochastic-approximation convergence conditions, $\varepsilon$-greedy action probabilities, softmax action selection, optimistic initial values, the UCB exploration bonus, and regret.

In a $k$-armed bandit the value of an action is $q(a)=\mathbb{E}[R_t\mid A_t=a]$ and $Q_t(a)$ is its estimate. Where a subscript counts the selections of a single arm we write $Q_n$ and $R_n$, as in the chapter.

### Exercise 4.1 — The incremental sample-average update

A single bandit arm returns the reward sequence $R_1,\dots,R_5 = 1,\,0,\,2,\,4,\,3$. Starting from $Q_1=0$:

1. Derive the incremental update rule for the sample average.
2. Apply it step by step to obtain $Q_2,\dots,Q_6$ and check the final value equals the plain average.

**Step 1 — Derive the incremental form.** The sample average after $n$ rewards is $Q_{n+1}=\frac1n\sum_{i=1}^n R_i$. Separating the last term:

$\displaystyle Q_{n+1} = \frac1n\Big(R_n+\sum_{i=1}^{n-1}R_i\Big) = \frac1n\big(R_n+(n-1)Q_n\big) = Q_n + \frac1n\big(R_n - Q_n\big).$

This is the general form **NewEstimate ← OldEstimate + StepSize·(Target − OldEstimate)** with step size $\tfrac1n$.

**Step 2 — Iterate** (target = the new reward $R_n$):

| $n$ | $R_n$ | update | $Q_{n+1}$ |
|---|---|---|---|
| 1 | 1 | $0+\tfrac11(1-0)$ | $1.0$ |
| 2 | 0 | $1+\tfrac12(0-1)$ | $0.5$ |
| 3 | 2 | $0.5+\tfrac13(2-0.5)$ | $1.0$ |
| 4 | 4 | $1.0+\tfrac14(4-1.0)$ | $1.75$ |
| 5 | 3 | $1.75+\tfrac15(3-1.75)$ | $2.0$ |

**Step 3 — Check.** The plain average is $\tfrac{1+0+2+4+3}{5}=\tfrac{10}{5}=2.0 = Q_6$. ✓

**Remark on $Q_1$.** The first step uses step size $\tfrac11=1$, so $Q_2 = Q_1 + 1\cdot(R_1-Q_1) = R_1$: the initial estimate is **erased by the first reward** and the value chosen for $Q_1$ is irrelevant here. Contrast this with Exercise 4.2, where a constant step size keeps a (decaying) trace of $Q_1$ forever, and with Exercise 4.7, where that persistence is exactly what we exploit.

**Key concept**

With step size $\tfrac1n$ the incremental rule reproduces the exact running mean using $O(1)$ memory and computation per step — no need to store the reward history. This is the prototype of every value-learning update in the course.

### Exercise 4.2 — Constant step size = exponential recency weighting

For the same reward sequence $1,0,2,4,3$ and $Q_1=0$, use a **constant** step size $\alpha=0.5$:

1. Compute $Q_2,\dots,Q_6$.
2. Show that $Q_6$ is an exponentially weighted average of the rewards; list the weights and verify they (with the initial-estimate weight) sum to $1$.

**Step 1 — Iterate** $Q_{n+1}=Q_n+\alpha(R_n-Q_n)$ with $\alpha=0.5$:

| $n$ | $R_n$ | $Q_{n+1}$ |
|---|---|---|
| 1 | 1 | $0.5$ |
| 2 | 0 | $0.25$ |
| 3 | 2 | $1.125$ |
| 4 | 4 | $2.5625$ |
| 5 | 3 | $2.78125$ |

**Step 2 — Weighted-average form.** Unrolling the recursion gives

$\displaystyle Q_{n+1} = (1-\alpha)^n Q_1 + \sum_{i=1}^{n}\alpha(1-\alpha)^{\,n-i} R_i.$

For $n=5,\ \alpha=0.5$, the weight on $R_i$ is $\alpha(1-\alpha)^{5-i}=0.5\cdot0.5^{5-i}$:

$\displaystyle w(R_5)=0.5,\ w(R_4)=0.25,\ w(R_3)=0.125,\ w(R_2)=0.0625,\ w(R_1)=0.03125,$

plus the weight on the initial estimate $(1-\alpha)^5 = 0.03125$.

**Step 3 — Checks.** Weights sum to $0.5+0.25+0.125+0.0625+0.03125+0.03125 = 1$ ✓. And the weighted sum reproduces $Q_6$:

$\displaystyle 0.5(3)+0.25(4)+0.125(2)+0.0625(0)+0.03125(1)+0.03125(0) = 2.78125 . ✓$

**Key concept**

A constant $\alpha$ makes recent rewards count exponentially more than old ones — ideal for **non-stationary** problems. The price is that the estimate never fully "forgets" the (decaying) influence of the initial guess and never completely converges, continuing to track the latest rewards.

### Exercise 4.3 — Convergence conditions for the step size

The Robbins–Monro conditions guarantee convergence of a stochastic-approximation estimate:

$\displaystyle \text{(C1)}\quad \sum_{n=1}^{\infty}\alpha_n = \infty, \qquad\qquad \text{(C2)}\quad \sum_{n=1}^{\infty}\alpha_n^2 < \infty.$

For each schedule below, state whether (C1) and (C2) hold, and hence whether convergence to the true action value is guaranteed:

$\displaystyle \text{(a)}\ \alpha_n=\tfrac1n, \qquad \text{(b)}\ \alpha_n=\tfrac{1}{n^2}, \qquad \text{(c)}\ \alpha_n=c\ (\text{constant}), \qquad \text{(d)}\ \alpha_n=\tfrac{1}{\sqrt{n}}.$

**Step 1 — Recall the convergence criterion.**

Recall the $\displaystyle p$-series fact: $\displaystyle\sum_n n^{-p}$ **diverges** for $p\le 1$ and **converges** for $p>1$. Check (C1) and (C2) against this fact for each schedule.

**Step 2 — Schedules (a) and (b): the decaying step sizes.**

**(a) $\alpha_n=1/n$.**
- (C1): $\sum 1/n$ is the harmonic series → **diverges** ✓.
- (C2): $\sum 1/n^2$ ($p=2>1$) → **converges** ✓.
- **Both hold → convergence guaranteed.** (This is the sample-average case.)

**(b) $\alpha_n=1/n^2$.**
- (C1): $\sum 1/n^2$ → **converges** ✗ (fails C1).
- Steps shrink too fast; the estimate can get "stuck" before overcoming its initial condition. **Not guaranteed.**

**Step 3 — Schedules (c) and (d): the non-decaying step sizes.**

**(c) $\alpha_n=c$ (constant).**
- (C1): $\sum c=\infty$ ✓.
- (C2): $\sum c^2=\infty$ → **diverges** ✗ (fails C2).
- The estimate never settles; it keeps fluctuating with recent rewards. **Not guaranteed** — but this is exactly what we *want* for non-stationary problems.

**(d) $\alpha_n=1/\sqrt{n}$.**
- (C1): $\sum n^{-1/2}$ ($p=\tfrac12\le1$) → **diverges** ✓.
- (C2): $\sum n^{-1}$ → **diverges** ✗ (fails C2).
- **Not guaranteed.**

**Step 4 — Summary.**

| schedule | C1 ($\sum\alpha_n=\infty$) | C2 ($\sum\alpha_n^2<\infty$) | converges? |
|---|---|---|---|
| $1/n$ | ✓ | ✓ | **yes** |
| $1/n^2$ | ✗ | ✓ | no |
| $c$ | ✓ | ✗ | no (good for non-stationary) |
| $1/\sqrt n$ | ✓ | ✗ | no |

**Key concept**

(C1) keeps the steps *large enough* to overcome initial bias and noise; (C2) makes them *small enough* eventually to settle. Only $\alpha_n=1/n$ satisfies both; constant $\alpha$ deliberately violates (C2) to stay adaptive. Note that failing the conditions is not the same as diverging: it only means the theorem does not apply, and Exercise 4.8 shows a case where we *want* it not to apply.

### Exercise 4.4 — $\varepsilon$-greedy action probabilities and expected reward

A $k=4$ armed bandit uses an $\varepsilon$-greedy policy with $\varepsilon=0.2$. The current estimates make action $a_4$ the unique greedy action. The **true** action values are $q(a_1),\dots,q(a_4) = 1,\,2,\,0,\,3$.

1. Compute the probability of selecting each action.
2. Compute the expected reward obtained on a single step under this policy.

**Step 1 — Selection probabilities.** Under $\varepsilon$-greedy the greedy action is chosen either by exploitation (prob. $1-\varepsilon$) or by landing on it during the random draw (prob. $\varepsilon/k$); every other action is chosen only via the random draw:

$\displaystyle \Pr(\text{greedy }a_4) = 1-\varepsilon+\frac{\varepsilon}{k} = 0.8 + \frac{0.2}{4} = 0.8 + 0.05 = 0.85,$
$\displaystyle \Pr(a_1)=\Pr(a_2)=\Pr(a_3) = \frac{\varepsilon}{k} = \frac{0.2}{4} = 0.05.$

Check: $0.85 + 3(0.05) = 1$ ✓.

**Step 2 — Expected reward per step.** $\ \mathbb{E}[R] = \sum_a \Pr(a)\,q(a)$:

$\displaystyle \mathbb{E}[R] = 0.85\,(3) + 0.05\,(1) + 0.05\,(2) + 0.05\,(0) = 2.55 + 0.15 = 2.7.$

The greedy action has the highest true value here, so the $0.15$ contributed by the other three terms is not a gain: the oracle that always plays $a_4$ would earn $3$, so $0.3$ per step is the **price paid for exploration**. Exercise 4.9 turns that observation into the definition of regret.

**Key concept**

$\varepsilon$-greedy never stops exploring: even after identifying the best action it selects it only with probability $1-\varepsilon+\varepsilon/k$, capping its exploitation. Decaying $\varepsilon$ over time recovers the best of both worlds.

### Exercise 4.5 — Upper Confidence Bound (UCB) selection

A $3$-armed bandit has been played $t=6$ times so far, with selection counts $N=(3,2,1)$ and value estimates $Q=(1.0,\,1.5,\,2.0)$ for arms $1,2,3$. Using the UCB rule with $c=2$,

$\displaystyle A_t=\arg\max_a\Big[\,Q_t(a) + c\sqrt{\tfrac{\ln t}{N_t(a)}}\,\Big],$

compute the UCB score of each arm and determine which arm is selected next. Comment on the role of the bonus.

*(Note the placement of $c$: it multiplies the square root, it is not inside it. This is the convention of `Notation.ipynb` and of the implementation in the chapter.)*

**Step 1 — Compute the exploration bonus** $c\sqrt{\ln t / N_t(a)}$ with $\ln 6 \approx 1.7918$:

$\displaystyle \text{arm 1: } 2\sqrt{\tfrac{1.7918}{3}} = 2\sqrt{0.5973}=1.5456,\quad \text{arm 2: } 2\sqrt{\tfrac{1.7918}{2}} = 1.8930,\quad \text{arm 3: } 2\sqrt{\tfrac{1.7918}{1}} = 2.6771.$

**Step 2 — Add the value estimate.**

| arm | $Q$ | bonus | UCB score |
|---|---|---|---|
| 1 | 1.0 | 1.5456 | $2.5456$ |
| 2 | 1.5 | 1.8930 | $3.3930$ |
| 3 | 2.0 | 2.6771 | $4.6771$ |

**Step 3 — Select.** The maximum UCB score is arm **3** ($4.6771$), which is selected next.

**Step 4 — Interpretation.** Arm 3 wins on *both* counts here: it has the highest estimate **and** the largest bonus (it is the least-tried, so the most uncertain). Even an arm with a lower estimate can be selected if its bonus is large enough — UCB gives the "benefit of the doubt" to under-sampled actions. As an arm is played, its $N$ grows and its bonus shrinks like $1/\sqrt{N}$ (while $\ln t$ grows only slowly), so attention naturally shifts.

**Key concept**

UCB is *optimism in the face of uncertainty*: it explores deterministically by adding a confidence bonus that is largest for the actions we know least about, rather than exploring blindly like $\varepsilon$-greedy.

### Exercise 4.6 — Softmax action selection and the temperature

A $3$-armed bandit has estimates $Q=(1,\,2,\,3)$. Softmax (Boltzmann) selection assigns

$\displaystyle \Pr(a) = \frac{e^{Q(a)/\tau}}{\sum_{b=1}^{k} e^{Q(b)/\tau}}.$

1. Compute the selection probabilities for $\tau=0.5$ and $\tau=2$.
2. Show that the ratio $\Pr(a_3)/\Pr(a_2)$ depends only on the **difference** $Q(a_3)-Q(a_2)$ and on $\tau$, and use it to explain the two limits $\tau\to0$ and $\tau\to\infty$.
3. Why does an implementation subtract $\max_b Q(b)$ before exponentiating?

**Step 1 — Probabilities.** With $\tau=0.5$ the exponents are $Q/\tau = (2,4,6)$:

$\displaystyle \Pr = \frac{(e^{2},\,e^{4},\,e^{6})}{e^{2}+e^{4}+e^{6}} = \frac{(7.389,\ 54.598,\ 403.429)}{465.416} = (0.0159,\ 0.1173,\ 0.8668).$

With $\tau=2$ the exponents are $Q/\tau=(0.5,1,1.5)$:

$\displaystyle \Pr = \frac{(1.6487,\ 2.7183,\ 4.4817)}{8.8487} = (0.1863,\ 0.3072,\ 0.5065).$

Both sum to $1$ ✓. The low temperature concentrates almost $87\%$ of the mass on the best arm; the high temperature is already close to the uniform $1/3$.

**Step 2 — Only differences matter.** Dividing the two expressions, the normalising constant cancels:

$\displaystyle \frac{\Pr(a_3)}{\Pr(a_2)} = \frac{e^{Q(a_3)/\tau}}{e^{Q(a_2)/\tau}} = e^{\,[Q(a_3)-Q(a_2)]/\tau}.$

Check: $\tau=0.5 \Rightarrow e^{1/0.5}=e^2=7.389 = 0.8668/0.1173$ ✓; $\tau=2 \Rightarrow e^{0.5}=1.649 = 0.5065/0.3072$ ✓.

The two limits follow immediately, for any fixed gap $\Delta Q>0$:

- $\tau\to 0$: the exponent $\Delta Q/\tau \to +\infty$, so the ratio diverges — **all the mass moves onto the greedy action**, and softmax degenerates into $\arg\max$ (pure exploitation).
- $\tau\to\infty$: the exponent $\to 0$, so every ratio $\to 1$ — **the distribution becomes uniform**, i.e. pure exploration.

This is why $\tau$ is the softmax counterpart of $\varepsilon$, and why decaying $\tau$ plays the role of decaying $\varepsilon$. Note the qualitative difference from $\varepsilon$-greedy: here the exploration is **value-sensitive**, a clearly bad arm is rarely tried, whereas $\varepsilon$-greedy explores all non-greedy arms equally.

**Step 3 — Numerical stability.** Since only differences matter, replacing $Q(a)$ by $Q(a)-\max_b Q(b)$ leaves every probability unchanged (numerator and denominator are both multiplied by $e^{-\max_b Q(b)/\tau}$). But it makes every exponent $\le 0$, so every exponential lies in $(0,1]$ and cannot overflow. Without it, a small $\tau$ combined with a moderate $Q$ overflows quickly: at $\tau=0.01$ and $Q=3$ we would evaluate $e^{300}$, well beyond the range of a 64-bit float. This is the **log-sum-exp trick**.

**Key concept**

Softmax turns estimates into a probability distribution whose shape is controlled by a single temperature: it explores *in proportion to how good an action looks*, unlike $\varepsilon$-greedy which explores uniformly. The same construction reappears later as the Boltzmann policy and as the parameterised policy of policy-gradient methods.

### Exercise 4.7 — Optimistic initial values as a pseudo-count

An arm is initialised optimistically with $Q_0=5$ and an initial count $n_0=10$, and is then updated with the usual rule $Q \leftarrow Q + \frac{1}{n}(R-Q)$, where $n$ is the counter *including* $n_0$. The arm turns out to be worthless and returns $R=0$ every time.

1. Show that after $m$ observed rewards the estimate is $Q = \dfrac{n_0Q_0 + \sum_{i=1}^{m}R_i}{n_0+m}$.
2. How many pulls are needed before the estimate falls below $4$? Below $1$?
3. Repeat with $n_0=1$ and comment on the role of $n_0$.

**Step 1 — The closed form.** This is Exercise 4.1 with a *prior*. Starting from $Q^{(0)}=Q_0$ with counter $n_0$, the $m$-th update uses step size $1/(n_0+m)$:

$\displaystyle Q^{(m)} = Q^{(m-1)} + \frac{1}{n_0+m}\big(R_m - Q^{(m-1)}\big) = \frac{(n_0+m-1)Q^{(m-1)} + R_m}{n_0+m}.$

By induction, if $Q^{(m-1)} = \frac{n_0Q_0+\sum_{i=1}^{m-1}R_i}{n_0+m-1}$ then the numerator becomes $n_0Q_0+\sum_{i=1}^{m-1}R_i + R_m$, giving

$\displaystyle Q^{(m)} = \frac{n_0Q_0+\sum_{i=1}^{m}R_i}{n_0+m}. \qquad\checkmark$

The interpretation is the useful part: the estimate is the sample average of the real rewards **plus $n_0$ imaginary rewards of value $Q_0$**. $n_0$ is a *pseudo-count*.

**Step 2 — How long the optimism survives.** With all $R_i=0$, the closed form reduces to $Q^{(m)} = \dfrac{50}{10+m}$:

| $m$ | 1 | 2 | 3 | 4 | 5 | 6 | 8 |
|---|---|---|---|---|---|---|---|
| $Q^{(m)}$ | $4.545$ | $4.167$ | $3.846$ | $3.571$ | $3.333$ | $3.125$ | $2.778$ |

- $Q<4 \iff 50 < 4(10+m) \iff m > 2.5$, so **3 pulls**.
- $Q<1 \iff 50 < 10+m \iff m > 40$, so **41 pulls**.

Note how slowly the estimate decays: after 8 disappointments the arm is still credited with a value of $2.78$, far above its true value of $0$. That is exactly the point — the agent keeps coming back.

**Step 3 — The role of $n_0$.** With $n_0=1$ the closed form is $Q^{(m)}=\dfrac{5}{1+m}$: $Q<4$ after a **single** pull, and $Q<1$ after $5$. The optimism evaporates almost immediately.

So $Q_0$ sets *how* optimistic we are and $n_0$ sets *how long the optimism lasts*. Together they determine the amount of exploration, and both are hand-tuned constants unrelated to the data — which is the objection raised in the chapter and answered by UCB, where the same role is played by $\sqrt{\ln t / N_t(a)}$, a quantity **computed from the observations** rather than guessed.

A further limitation follows from the same formula: $Q^{(m)}\to$ the true mean as $m$ grows, whatever $n_0$ is. The drive to explore is therefore **inherently temporary**, which makes optimistic initialization unsuitable for non-stationary problems.

**Key concept**

Optimistic initialization is a prior in disguise: $(Q_0, n_0)$ is equivalent to having already seen $n_0$ rewards of value $Q_0$. It buys an initial burst of exploration for free, but the burst is finite and its size is a hyperparameter, not an estimate of uncertainty.

### Exercise 4.8 — Tracking a non-stationary arm

An arm's true value is $0$ for a while and then **jumps to $10$**. We observe the reward sequence

$\displaystyle R_1,\dots,R_8 = 0,\,0,\,0,\,0,\,10,\,10,\,10,\,10 ,$

and start from $Q_1=0$.

1. Track the estimate with the sample average ($\alpha_n=1/n$) and with a constant step size ($\alpha=0.5$).
2. For each, how many of the *new* rewards are needed before the estimate exceeds $9$?
3. Relate the answer to the Robbins–Monro conditions of Exercise 4.3.

**Step 1 — The two traces.**

| $n$ | $R_n$ | $Q_{n+1}$ with $\alpha_n=1/n$ | $Q_{n+1}$ with $\alpha=0.5$ |
|---|---|---|---|
| 1 | 0 | $0$ | $0$ |
| 2 | 0 | $0$ | $0$ |
| 3 | 0 | $0$ | $0$ |
| 4 | 0 | $0$ | $0$ |
| 5 | 10 | $2.0$ | $5.0$ |
| 6 | 10 | $3.333$ | $7.5$ |
| 7 | 10 | $4.286$ | $8.75$ |
| 8 | 10 | $5.0$ | $9.375$ |

The constant step size has essentially caught up with the new value after four observations; the sample average is still at half of it.

**Step 2 — Time to reach $9$.**

*Constant $\alpha=0.5$*: the error is multiplied by $(1-\alpha)$ at every step, $10 - Q = 10\cdot 0.5^{\,j}$ after $j$ new rewards. We need $10\cdot0.5^{\,j} < 1 \iff j > \log_2 10 = 3.32$, i.e. **$j=4$** new rewards — matching the table.

*Sample average*: after $m$ total rewards (the first four of which are zeros) the estimate is $Q=\frac{10(m-4)}{m}$. Requiring $Q>9$:

$\displaystyle \frac{10(m-4)}{m} > 9 \iff 10m-40 > 9m \iff m > 40,$

so $m=41$ total rewards, i.e. **$j=37$** new rewards — roughly **nine times slower**, and the gap widens the later the jump occurs: if the change had happened after $1000$ zeros, the sample average would need about $9000$ new rewards while the constant step size would still need $4$.

**Step 3 — Reading it through the convergence conditions.** The sample average satisfies both Robbins–Monro conditions, so it converges — but it converges to the average over the **whole history**, which is the right target only if the target never moves. Its step size $1/n \to 0$, so an old agent is an agent that has stopped learning.

The constant step size violates (C2) on purpose. The consequence is exactly the trade-off seen here: it never converges (its estimate keeps fluctuating around the truth with a variance proportional to $\alpha$), but it never stops tracking either. The choice of $\alpha$ is the choice of a compromise between **noise rejection** (small $\alpha$) and **speed of adaptation** (large $\alpha$).

**Key concept**

Satisfying the convergence conditions is not a goal in itself: they guarantee convergence to a *fixed* target. In RL the targets keep moving — the environment may drift and, more importantly, the policy itself keeps changing — so a constant step size is the default choice throughout the course.

### Exercise 4.9 — Regret of an $\varepsilon$-greedy agent

Take again the bandit of Exercise 4.4: $k=4$, true values $q=(1,\,2,\,0,\,3)$, and an $\varepsilon$-greedy policy with $\varepsilon=0.2$ whose estimates have already identified $a_4$ as greedy and keep it greedy forever.

1. Compute the gaps $\Delta_a=q(a_*)-q(a)$ and the per-step regret.
2. Compute the total regret $L_T$ after $T=1000$ steps, using both

$\displaystyle L_T = \sum_{t=1}^{T}\big(q(a_*)-q(A_t)\big) \qquad\text{and}\qquad L_T=\sum_a \mathbb{E}[N_T(a)]\,\Delta_a ,$

and verify the two agree.
3. What is the asymptotic growth of $L_T$, and what would change with a decaying $\varepsilon$?

**Step 1 — Gaps and per-step regret.** The optimal arm is $a_4$ with $q(a_*)=3$, so

$\displaystyle \Delta_1 = 2,\quad \Delta_2 = 1,\quad \Delta_3 = 3,\quad \Delta_4 = 0 .$

With the selection probabilities of Exercise 4.4, $\Pr(a_4)=0.85$ and $\Pr(a_1)=\Pr(a_2)=\Pr(a_3)=0.05$, the expected loss on a single step is

$\displaystyle \mathbb{E}[\Delta_{A_t}] = 0.05(2) + 0.05(1) + 0.05(3) + 0.85(0) = 0.1+0.05+0.15 = 0.3 .$

Consistency check with Exercise 4.4: the expected reward there was $2.7$, and the oracle collects $q(a_*)=3$, so the per-step loss is $3-2.7=0.3$ ✓.

**Step 2 — Total regret over $T=1000$ steps.**

*First form.* The per-step regret is constant here, so $L_{1000} = 1000 \times 0.3 = \mathbf{300}$.

*Second form.* The expected counts are $\mathbb{E}[N_T(a)] = T\Pr(a)$, i.e. $(50,\,50,\,50,\,850)$:

$\displaystyle L_{1000} = 50(2) + 50(1) + 50(3) + 850(0) = 100+50+150 = \mathbf{300}. \ ✓$

The second form is the more informative one: it says the regret is *how often we played each bad arm* times *how bad it is*. Note that arm 3, the worst one, contributes half the total even though it is played as rarely as the others.

**Step 3 — Growth rate.** The per-step regret is a constant $\tfrac{\varepsilon}{k}\sum_a\Delta_a = 0.3$ that does not depend on $T$, so

$\displaystyle L_T = 0.3\,T = \Theta(T):$ the regret is **linear**.

A linear regret means $L_T/T \not\to 0$: no matter how long it runs, the agent keeps losing $0.3$ per step. In this precise sense a fixed-$\varepsilon$ agent **never finishes learning the problem**, even though its estimates converge perfectly — the loss comes not from ignorance but from the exploration it is contractually obliged to keep doing.

With a **decaying** $\varepsilon_t$ the per-step regret becomes $\frac{\varepsilon_t}{k}\sum_a \Delta_a$, which vanishes. A schedule such as $\varepsilon_t \propto 1/t$ gives $L_T \propto \sum_{t\le T} 1/t \approx \ln T$: **logarithmic**, hence sublinear. UCB achieves the same $\Theta(\ln T)$ rate without a schedule to tune, and Lai and Robbins proved that $\Omega(\ln T)$ is the best any strategy can do — so on this measure decaying $\varepsilon$-greedy and UCB are both optimal *up to constants*, which is what the chapter's parameter study is really measuring.

**Key concept**

Regret is the standard yardstick for exploration: it is zero for an oracle, and its **growth rate** classifies a strategy. Linear regret = a constant tax paid forever (pure exploitation, pure exploration, fixed $\varepsilon$); logarithmic regret = the tax vanishes (decaying $\varepsilon$, UCB), and $\ln T$ is provably the best possible.